# JEM application: Coupled Spring System


In this tutorial, we are going to build a two springs that interacts with each other. 

## Single Spring

Imagine first there is a metal ball with mass $m$ attached to a spring such that it feels the restoring force $- k \left(x - X\right)$, where $k$ is the restoring coefficient, $x$ the position of the metal ball, and $X$ the restoring center. The governing equation is
$$
m \frac{\mathrm{d}^2 x}{\mathrm{d}t^2} = - k \left(x - X\right)
$$
or let $\Delta x = x - X$, we can simplify it to
$$
m \frac{\mathrm{d}^2 \Delta x}{\mathrm{d}t^2} = - k \Delta x
$$
We can build such a spring with the following (define $v = \mathrm{d} \Delta x/\mathrm{d}t$):

In [ ]:
from dataclasses import dataclass

import jax.numpy as jnp
from jax.typing import ArrayLike
import jax_datetime as jdt
import tree_math

from jem.base.coupler import Coupler


@tree_math.struct
@dataclass
class SpringCarry:
    x: ArrayLike  # displacement from the restoring centre
    v: ArrayLike  # velocity
    m: ArrayLike  # mass
    k: ArrayLike  # spring coefficient
    f: ArrayLike  # external force, written by the exchanger


class Spring:
    """One spring, as a JEM component.

    A component is any object with three things: a `name`, an `initialize()`
    returning its initial carry, and a `step(carry, time)` advancing it by one
    coupling timestep. There is no base class to inherit from. It holds no
    clock of its own -- the coupler owns the one clock and hands it in as
    `time`.
    """

    def __init__(self, name, init_x, init_v, k, m):
        self.name = name
        self.init_x = init_x
        self.init_v = init_v
        self.k = k
        self.m = m

    def initialize(self):
        return SpringCarry(
            x=jnp.array(self.init_x, dtype=float),
            v=jnp.array(self.init_v, dtype=float),
            m=jnp.array(self.m, dtype=float),
            k=jnp.array(self.k, dtype=float),
            f=jnp.array(0.0, dtype=float),
        )

    def step(self, carry, time):
        """Integrate one time step of a harmonic oscillator."""
        # Physics: a = -(k x + f) / m
        acceleration = - (carry.k * carry.x + carry.f) / carry.m

        # Update state (semi-implicit Euler for better stability). The carry is
        # rebuilt rather than assigned into: `lax.scan` carries this object, so
        # writing into it would be a side effect on traced values.
        new_v = carry.v + acceleration * time.dt
        new_x = carry.x + new_v * time.dt
        new_carry = carry.replace(x=new_x, v=new_v)

        return new_carry, dict(
            t=time.end_of_step().sim_time,
            x=new_x,
            v=new_v,
            f=carry.f,
        )

Even a single component is run through the `Coupler`: it owns the clock, and
drives the component's `step` with `jax.lax.scan`.

The coupling timestep is a `jax_datetime.Timedelta`, which counts whole
seconds, so this example uses a one-second coupling step and spring constants
in units where that is a small step. With $k/m = 10^{-4}\,\mathrm{s^{-2}}$ the
oscillator turns through 0.01 radians per step -- the same discrete system as
$k=5$, $m=5$ integrated with $\Delta t = 0.01\,\mathrm{s}$, with its time axis
stretched by a factor of 100. The displacements are therefore identical; the
velocities, being per unit time, are 100 times smaller.

The rescaling is a workaround, not a property of the physics: the coupler
cannot yet be given a sub-second timestep. Tracked as jax-esm#110.


In [ ]:
import matplotlib.pyplot as plt

start_date = jdt.to_datetime("2000-01-01")
coupling_timestep = jdt.to_timedelta(1, "second")

model = Coupler(
    {"spring": Spring(name="spring", init_x=0, init_v=0.02, k=5.0e-4, m=1.0)},
    coupling_timestep=coupling_timestep,
    start_date=start_date,
)
run = model.generate_trajectory_function(1000)
final_carry, diagnostics = run(model.initialize())

predictions = diagnostics["spring"]
x = predictions["x"]
v = predictions["v"]
t = predictions["t"]

fig, ax = plt.subplots(1,1)
ax.plot(t, x, label="x")
ax.plot(t, v, label="v")
ax.legend()
ax.grid()
plt.show()

## Two Coupled Springs

Now we want to simulate two springs that interacts with each other.

\begin{align}
m_1 \frac{\mathrm{d}^2 \Delta x_1}{\mathrm{d}t^2} &= - k_1 \Delta x_1 + k^* \left( \Delta x_2 - \Delta x_1 \right) \\
m_2 \frac{\mathrm{d}^2 \Delta x_2}{\mathrm{d}t^2} &= - k_2 \Delta x_2 - k^* \left( \Delta x_2 - \Delta x_1 \right)
\end{align}

where 1 and 2 denote springs 1 and 2, $\Delta x_1 = x_1 - X_1$ and $\Delta x_2 = x_2 - X_2$.

Here we define the *exchanger*: the one place where components exchange
information. It receives the mapping of every component's carry together with
the coupler's clock, and returns the mapping to continue with -- building new
carries rather than writing into the ones it was handed.

In [ ]:
k_star = 1.0e-4


def interaction(components, time):
    del time  # this exchange does not depend on the date

    spring1 = components["spring1"]
    spring2 = components["spring2"]
    f_star = (spring2.x - spring1.x) * k_star

    return dict(
        components,
        spring1=spring1.replace(f=f_star),
        spring2=spring2.replace(f=-f_star),
    )

### Using JEM to Couple Springs

In [ ]:
model = Coupler(
    dict(
        spring1=Spring(name="spring1", init_x=0, init_v=0.04, k=5.0e-4, m=5.0),
        spring2=Spring(name="spring2", init_x=4, init_v=0, k=5.0e-4, m=3.0),
    ),
    dict(interaction=interaction),
    coupling_timestep=coupling_timestep,
    start_date=start_date,
)

# The workflow defaults to every exchanger followed by every component, i.e.
# ("interaction", "spring1", "spring2") -- the springs feel the force computed
# from the previous step's positions.
print(repr(model))

run = model.generate_trajectory_function(5000)
final_coupled_carry, predictions = run(model.initialize())

### Results

In [ ]:
import matplotlib.pyplot as plt

x1 = predictions["spring1"]["x"]
v1 = predictions["spring1"]["v"]
x2 = predictions["spring2"]["x"]
v2 = predictions["spring2"]["v"]
f1 = predictions["spring1"]["f"]
f2 = predictions["spring2"]["f"]
t = predictions["spring1"]["t"]

# Figure 1: x-t 
fig, ax = plt.subplots(3,1, figsize=(10,10))
ax[0].plot(t, x1, label="x1")
ax[0].plot(t, x2, label="x2")
ax[1].plot(t, v1, label="v1")
ax[1].plot(t, v2, label="v2")
ax[2].plot(t, f1, label="f1")
ax[2].plot(t, f2, label="f2")

ax[0].legend()
ax[1].legend()
ax[2].legend()

ax[0].grid()
ax[1].grid()
ax[2].grid()

for _ax in ax:
    _ax.set_xlabel("Time [s]")
    
ax[0].set_ylabel("$\\Delta x$ [m]")
ax[1].set_ylabel("Velocity $ v$ [m/s]")
ax[2].set_ylabel("External force [N]")

# Figure 2: x-v
fig, ax = plt.subplots(2,1)
ax[0].plot(x1, x2, label="x")
ax[1].plot(v1, v2, label="v")

ax[0].legend()
ax[1].legend()

ax[0].grid()
ax[1].grid()

ax[0].set_xlabel("$\\Delta x_1$ [m]")
ax[0].set_ylabel("$\\Delta x_2$ [m]")

ax[1].set_xlabel("$v_1$ [m/s]")
ax[1].set_ylabel("$v_2$ [m/s]")


plt.show()